# Step 1: Load Official Dataset

**Goal**: Load official CSV and save to Delta table

- Dynamically detect username (judge-reproducible)
- Load CSV from data folder
- Save to `workspace.bronze.fraud_data`

In [0]:
# Get current user dynamically (JUDGE-REPRODUCIBLE)
import pandas as pd
import os

# Detect username automatically
try:
    username = spark.sql("SELECT current_user()").collect()[0][0]
except:
    username = os.environ.get('USER', 'default_user')

print(f"Current user: {username}")

# Build paths dynamically
csv_path = f"/Workspace/Users/{username}/Suraksha/data/official_dataset.csv"
print(f"\nCSV path: {csv_path}")

In [0]:
# Load CSV
print("Loading CSV...")
df = pd.read_csv(csv_path)

print(f"✓ Loaded {len(df):,} transactions")
print(f"✓ Fraud cases: {df['fraud_flag'].sum()} ({df['fraud_flag'].sum()/len(df)*100:.2f}%)")

In [0]:
# Rename columns
df = df.rename(columns={
    'transaction id': 'txn_id',
    'transaction type': 'txn_type',
    'amount (INR)': 'amount_inr',
    'transaction_status': 'txn_status',
    'fraud_flag': 'is_fraud'
})

print("✓ Renamed columns")
df.head()

In [0]:
# Save to Delta
df_spark = spark.createDataFrame(df)
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.bronze")
df_spark.write.format("delta").mode("overwrite").saveAsTable("workspace.bronze.fraud_data")

print("✓ Saved to workspace.bronze.fraud_data")